# Early Stress Intervention Orchestrator## Notebook 2 of 3 — Agent CodeDefines and tests the four core agents:1. **Triage Agent** — decides MONITOR / EMAIL / CALL / ESCALATE2. **Email Sub-Agent** — drafts and sends personalised emails via SendGrid3. **Voice Sub-Agent** — places outbound AI calls via Vapi.ai4. **Summariser Agent** — extracts structured intelligence from transcripts5. **Reporter Agent** — writes the autonomous weekly executive memoPlus a `generate_mock_transcript` helper for the demo (Vapi free tiercan't dial Australia).**Run cells top to bottom.** Each agent has a built-in test at the end.

In [ ]:
# Cell 1 — Setupfrom google.colab import drivedrive.mount('/content/drive')!pip install groq supabase sendgrid requests python-dotenv -q# Fill these in from each service's dashboardSUPABASE_URL  = "your-supabase-url"SUPABASE_KEY  = "your-anon-key"GROQ_API_KEY  = "your-groq-key"SENDGRID_KEY  = "your-sendgrid-key"SENDGRID_FROM = "your-verified-sender@example.com"YOUR_EMAIL    = "your-personal-email@example.com"VAPI_KEY      = "your-vapi-key"VAPI_PHONE    = "your-vapi-phone-id"  # UUID, not the phone number itselfTEST_PHONE    = "+61XXXXXXXXX"RAILWAY_URL   = "your-railway-url"    # fill in after deploying main.pyMODEL = "llama-3.3-70b-versatile"print("Setup complete")

In [ ]:
# Cell 2 — Triage Agentfrom groq import Groqfrom supabase import create_clientimport jsongroq_client = Groq(api_key=GROQ_API_KEY)sb = create_client(SUPABASE_URL, SUPABASE_KEY)def triage_customer(customer: dict, signals: list) -> dict:    """Decide what action to take for a single customer."""    prompt = f"""You are a credit risk analyst at an Australian bank.Assess this customer and decide on an action.Customer: {json.dumps(customer, indent=2, default=str)}Last 4 weeks signals: {json.dumps(signals[-4:], indent=2, default=str)}Respond ONLY in valid JSON:{{  "stress_assessment": "low | moderate | high | critical",  "key_signals": ["2-3 signals driving your view"],  "reasoning": "2 sentences explaining your conclusion",  "action": "MONITOR | EMAIL | CALL | ESCALATE",  "action_rationale": "why this action and not another",  "urgency": "routine | soon | immediate"}}Rules: MONITOR for most customers. EMAIL for moderate stress withearly warning signs. CALL for high/critical only. ESCALATE onlyfor imminent default risk."""    resp = groq_client.chat.completions.create(        model=MODEL,        messages=[{"role": "user", "content": prompt}],        response_format={"type": "json_object"},        temperature=0.1    )    return json.loads(resp.choices[0].message.content)# Test ittest_cust = sb.table('customers').select('*')\    .eq('risk_tier', 'Severe').limit(1).execute().data[0]test_sigs = sb.table('weekly_signals').select('*')\    .eq('customer_id', test_cust['customer_id']).execute().dataresult = triage_customer(test_cust, test_sigs)print(json.dumps(result, indent=2))

In [ ]:
# Cell 3 — Email Sub-Agent (SendGrid)import sendgridfrom sendgrid.helpers.mail import Maildef send_email_intervention(customer: dict, triage: dict) -> dict:    """Draft a personalised email via Groq and send via SendGrid."""    first = customer['name'].split()[0]    # Step 1 — Groq drafts the email    draft = groq_client.chat.completions.create(        model=MODEL,        messages=[{"role": "user", "content": f"""Write a warm email from CommBank to {first}.They may be under some financial pressure but don't know we know.Purpose: offer a free home loan health check call with a specialist.Rules:- Subject line: short, curiosity-driven, never alarming- Body: 3 short paragraphs. Warm and human, not corporate- Never mention risk, stress, arrears, or internal bank data- End with two options: "Book a call" or "I'm fine, no thanks"- Sign off as: Maya Chen, Home Loan Specialist, CommBankRespond in JSON: {{"subject": "...", "body_text": "..."}}"""}],        response_format={"type": "json_object"},        temperature=0.7    )    content = json.loads(draft.choices[0].message.content)    subject = content['subject']    body = content['body_text']    # Step 2 — Send via SendGrid (to YOUR_EMAIL during dev/demo)    sg = sendgrid.SendGridAPIClient(api_key=SENDGRID_KEY)    message = Mail(        from_email=SENDGRID_FROM,        to_emails=YOUR_EMAIL,        subject=f"[TEST - {customer['customer_id']}] {subject}",        plain_text_content=body    )    response = sg.send(message)    # Step 3 — Mark sent in DB    sb.table('customers').update({'email_sent': True})\        .eq('customer_id', customer['customer_id']).execute()    print(f"  Email sent -> {YOUR_EMAIL}")    print(f"  Subject: {subject}")    print(f"  Status: {response.status_code}")    return {"subject": subject, "body": body, "status": response.status_code}# Test it (sends a real email to YOUR_EMAIL)email_result = send_email_intervention(test_cust, result)

In [ ]:
# Cell 4 — Voice Sub-Agent (Vapi.ai)## Note: Vapi free-tier accounts can only place outbound calls to US# numbers. For the demo, voice interventions are populated via# generate_mock_transcript() (next cell) and processed by the real# Summariser. The place_call() function below is preserved to show# the production integration code.import requestsdef place_call(customer: dict, triage: dict) -> dict:    """Place an outbound AI voice call via Vapi.ai."""    first = customer['name'].split()[0]    resp = requests.post(        "https://api.vapi.ai/call/phone",        headers={"Authorization": f"Bearer {VAPI_KEY}"},        json={            "phoneNumberId": VAPI_PHONE,            "customer": {"number": TEST_PHONE},            "metadata": {"customer_id": customer['customer_id']},            "assistant": {                "name": "Maya",                "firstMessage": f"Hi {first}, it's Maya from CommBank. Got a minute?",                "model": {                    "provider": "groq",                    "model": MODEL,                    "systemPrompt": f"""You are Maya, a home loan specialist at CommBank.Calling {first} for a routine wellbeing check.Can offer: repayment pause (3 months), interest-only (6 months),or hardship team referral.Listen for: job changes, medical costs, reduced hours.Be warm and human. Never mention risk scores."""                },                "voice": {"provider": "playht", "voiceId": "jennifer"},                "serverUrl": f"{RAILWAY_URL}/vapi-webhook"            }        }    )    return resp.json()print("place_call() defined")print("Vapi free tier limits outbound calls to US numbers — use generate_mock_transcript() for AU demos.")

In [ ]:
# Cell 5 — Mock transcript generator (for demo data)## Generates realistic call transcripts using Groq so the dashboard can# show populated voice-intervention data without using Vapi credits.def generate_mock_transcript(customer: dict) -> str:    """Generate a realistic 2-minute call transcript."""    first = customer['name'].split()[0]    prompt = f"""Generate a realistic 2-minute phone conversation transcriptbetween Maya (a CommBank home loan specialist) and {first}, a customershowing mortgage stress signals.Format strictly as:Maya: ...{first}: ...Maya: ...Maya should: greet warmly, ask how things are going, listen for stresssignals, offer one of: repayment pause, interest-only switch, orhardship referral.The customer should: be initially cautious, then disclose a real-lifereason (reduced hours, medical expense, family change), then accept ordecline an offer.End with a clear next step.Make it feel real. Use natural Australian speech patterns."""    response = groq_client.chat.completions.create(        model=MODEL,        messages=[{"role": "user", "content": prompt}],        temperature=0.8    )    return response.choices[0].message.content# Test itsample = generate_mock_transcript(test_cust)print(sample)

In [ ]:
# Cell 6 — Summariser Agentdef summarise_interaction(transcript: str, customer: dict, channel: str) -> dict:    """Extract structured intelligence from a transcript or email exchange."""    channel_context = {        "email": "outbound email and customer reply",        "voice call": "outbound phone call transcript",        "escalation": "internal escalation note"    }.get(channel, channel)    prompt = f"""You are a credit risk analyst reviewing a {channel_context}.Customer: {customer['name']}, risk tier: {customer['risk_tier']}Content:{transcript}Respond ONLY in valid JSON:{{  "sentiment": "positive | neutral | concerned | distressed",  "engaged": true or false,  "reasons_disclosed": ["any reasons customer mentioned"],  "offer_accepted": "none | repayment_pause | interest_only | hardship_referral | booked_call",  "outcome": "resolved | monitor | follow_up | escalate_to_human",  "new_risk_signals": ["new signals learned"],  "updated_stress_assessment": "low | moderate | high | critical",  "next_action": "what happens next triage cycle",  "rm_briefing": "one paragraph for human RM if escalating, else null"}}"""    resp = groq_client.chat.completions.create(        model=MODEL,        messages=[{"role": "user", "content": prompt}],        response_format={"type": "json_object"},        temperature=0.1    )    return json.loads(resp.choices[0].message.content)# Test with the mock transcript we just generatedsummary = summarise_interaction(sample, test_cust, 'voice call')print(json.dumps(summary, indent=2))

In [ ]:
# Cell 7 — Reporter Agentdef generate_weekly_report(week: int, interventions: list) -> str:    """Generate a CBA-style executive credit risk memo for the week."""    resolved  = [i for i in interventions if i.get('outcome') == 'resolved']    escalated = [i for i in interventions if i.get('outcome') == 'escalate_to_human']    emails    = [i for i in interventions if i.get('channel') == 'email']    calls     = [i for i in interventions if i.get('channel') == 'voice']    accepted  = [i for i in interventions if i.get('offer_accepted', 'none') != 'none']    prompt = f"""Write a weekly credit risk executive briefing memo.Week {week} data:- Total interventions: {len(interventions)}- Email interventions: {len(emails)} | Voice calls: {len(calls)}- Resolved autonomously: {len(resolved)}- Escalated to human RM: {len(escalated)}- Offer/call acceptance rate: {len(accepted)}/{len(interventions)}Write 3 short paragraphs as a CBA credit risk memo:1. Portfolio stress summary and channel mix this week2. Email and voice intervention effectiveness3. Recommended focus areas for next weekBe specific with numbers. Professional tone. No fluff."""    resp = groq_client.chat.completions.create(        model=MODEL,        messages=[{"role": "user", "content": prompt}]    )    return resp.choices[0].message.content# Test itfake_interventions = [    {'outcome': 'resolved', 'channel': 'email', 'offer_accepted': 'booked_call'},    {'outcome': 'resolved', 'channel': 'voice', 'offer_accepted': 'repayment_pause'},    {'outcome': 'escalate_to_human', 'channel': 'escalation', 'offer_accepted': 'none'},]report = generate_weekly_report(1, fake_interventions)print(report)

# Done!## All five agents are defined and tested. Open notebook 03_orchestrator.ipynb# next to run the full Monday-morning intervention loop across multiple# weeks of customers.